# Policy Assistant Agent (Single Agent + Memory + Graph Orchestration)

**Week 27 Graded Mini Project** - IITM Pravartak, Professional Certificate Programme in Generative AI and Machine Learning

This notebook drives the modular repository in this folder (`models.py`, `tools.py`, `prompts.py`, `agent.py`, `memory.py`, `graph.py`, `eval.py`, `app.py`, `tests/`) and records every output: tests, evaluation tables, logs, a cold run, a warm run, error recovery, and the PNG results in `results/`.

- **Model:** OpenAI `gpt-4o-mini` (temperature 0, seed 42), read from `.env`.
- **Everything else is local and open-source:** FAISS on CPU, MiniLM embeddings, and SQLite for the LLM cache and checkpoints.

## 0. Setup

In [ ]:
import os, sys, json, subprocess, textwrap, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")      # library deprecation notices include install paths; keep outputs clean
from importlib.metadata import version

# OFFLINE=1 runs everything with the offline stand-in model (no key needed) and writes to runs_offline/ +
# results_offline/. The graded run uses the real OpenAI model (OFFLINE unset).
OFFLINE = os.getenv("POLICY_OFFLINE") == "1"
os.environ["POLICY_AGENT_RUN_DIR"] = str(Path("runs_offline" if OFFLINE else "runs").resolve())
RES = Path("results_offline" if OFFLINE else "results"); RES.mkdir(exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

import agent, graph, memory, tools, prompts, eval as ev
from fake_llm import OfflinePolicyLLM
LLM = OfflinePolicyLLM() if OFFLINE else None          # None -> agent.get_llm() (OpenAI)
CLI_FLAG = ["--offline"] if OFFLINE else []

print("MODE:", "OFFLINE stand-in model" if OFFLINE else f"LIVE OpenAI {agent.OPENAI_MODEL}")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print({p: version(p) for p in ["langchain", "langchain-classic", "langgraph", "langchain-openai", "faiss-cpu",
                               "sentence-transformers", "pydantic"]})
print("run dir:", agent.rel(agent.RUN_DIR))

In [ ]:
import re
ANSI = re.compile(r"\[[0-9;]*m")

def term_png(text, path, title):
    """Save a terminal transcript as a PNG 'screenshot' (dark terminal style)."""
    lines = []
    for l in ANSI.sub("", text).rstrip().splitlines():
        lines += textwrap.wrap(l, 125, subsequent_indent="    ") or [""]
    lines = lines[:70]
    fig = plt.figure(figsize=(13, 0.16 * len(lines) + 0.6))
    fig.patch.set_facecolor("#1e1e1e")
    fig.text(0.01, 0.995, title, color="#4fc1ff", family="monospace", fontsize=10, va="top", weight="bold")
    fig.text(0.01, 0.995 - 0.35 / (0.16 * len(lines) + 0.6), "\n".join(lines), color="#d4d4d4",
             family="monospace", fontsize=8, va="top")
    fig.savefig(path, dpi=110, facecolor=fig.get_facecolor()); plt.close(fig)

def cli(*args, title=None, png=None):
    """Run app.py like a user would in a terminal; print and optionally screenshot the output."""
    cmd = [sys.executable, "app.py", *args, *CLI_FLAG]
    out = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "PYTHONWARNINGS": "ignore"})
    out.stdout, out.stderr = ANSI.sub("", out.stdout), ANSI.sub("", out.stderr)
    text = "$ python app.py " + " ".join(f'"{a}"' if " " in a else a for a in args) + "\n" + "\n".join(
        l for l in (out.stdout + out.stderr).splitlines()
        if not any(s in l for s in ["Warning", "warn(", "Loading weights", "validate_python", "redirects", "sunset", "from langchain", "return [loads"]))
    print(text)
    if png:
        term_png(text, RES / png, title or png)
    return text

# Part 1: Foundations

## 1.1 Data prep: `data/policies.jsonl`

In [ ]:
pol = pd.read_json("data/policies.jsonl", lines=True)
pol["words"] = pol["text"].str.split().str.len()
print(len(pol), "policies; fields:", list(pol.columns[:3]))
pol[["doc_id", "date", "words", "text"]]

The corpus has 8 short policies of a **fictional** company, each with a `doc_id` (with a version suffix), `text` and `date`. `remote_work_v1` is deliberately kept as a *superseded* older version, to test whether the agent prefers the latest policy, as acceptance rule 4 requires.

## 1.2 Tools: contracts, typed errors and soft failure

In [ ]:
print("policy_search ->", tools.policy_search.invoke({"query": "Is MFA required for VPN?", "k": 2})[:300], "...\n")
print("read_policy(leave_v3) ->", tools.read_policy.invoke({"doc_id": "leave_v3"})[:160], "...\n")
try:
    tools.read_policy_text("remote_work_v9.9")
except tools.PolicyNotFoundError as e:
    print("typed error raised by read_policy_text:", type(e).__name__, "| suggestions:", e.suggestions)
print("tool wrapper fails softly ->", tools.read_policy.invoke({"doc_id": "remote_work_v9.9"}))
print("calculator('12 * 4') ->", tools.calculator.invoke({"expression": "12 * 4"}))
print("calculator('5 / 0') ->", tools.calculator.invoke({"expression": "5 / 0"}))
print("calculator('__import__(\"os\")') ->", tools.calculator.invoke({"expression": "__import__('os')"}))

## 1.3 Prompts: system prompt with acceptance rules

In [ ]:
print(prompts.SYSTEM_PROMPT)

## 1.4 Agent: deterministic ReAct tool-calling agent with verbose tracing

`create_tool_calling_agent` + `AgentExecutor` runs with `max_iterations=6`, `temperature=0`, `seed=42` and `verbose=True`. The trace below shows **bad-id recovery**. `read_policy("remote_work_v9.9")` fails softly with a typed `PolicyNotFoundError` message, so the agent calls `policy_search` and then reads the correct, latest policy.

In [ ]:
executor = agent.build_agent(LLM, verbose=True)
res = agent.run_agent("Read policy remote_work_v9.9 and answer the limit.", executor=executor)
print("\nFINAL:", res.answer)
print("steps:", res.steps, "| tools:", [c["tool"] for c in res.tool_calls], "| latency:", res.latency_s, "s")

## 1.5 Evaluation Version 1: the agent alone (before memory, graph and critic)

In [ ]:
v1 = ev.run_agent_eval(agent.build_agent(LLM, verbose=False))
print(v1[["id", "kind", "latency_s", "agent_steps", "has_docid", "has_quote", "cited", "PASS"]].to_string(index=False))
print(f"\nV1 pass rate: {v1['PASS'].mean():.0%}  | logs saved in", agent.rel(agent.RUN_DIR / "logs"))

# Part 2: Memory + Graph + Critic

## 2.2 The graph

The graph runs `plan → memory_read → agent_answer → critic → memory_write → stop`. A router after each node enforces the **graph budget** (`max_steps = 8`). The router after `critic` allows **one retry** when the critic rejects an answer.

In [ ]:
import matplotlib.patches as mp
def draw_graph(path):
    fig, ax = plt.subplots(figsize=(12, 3.6)); ax.axis("off")
    nodes = {"plan": (0.6, 2), "memory_read": (2.4, 2), "agent_answer": (4.4, 2), "critic": (6.3, 2),
             "memory_write": (8.2, 2), "stop": (10.0, 2), "retry\n(retries < 1)": (5.35, 0.45)}
    for n, (x, y) in nodes.items():
        c = "#ffd8a8" if n == "critic" else "#d0ebff" if "retry" not in n else "#ffc9c9"
        ax.add_patch(mp.FancyBboxPatch((x - 0.75, y - 0.35), 1.5, 0.7, boxstyle="round,pad=0.05", fc=c, ec="#333"))
        ax.text(x, y, n, ha="center", va="center", fontsize=9, weight="bold")
    arrow = dict(arrowstyle="->", lw=1.4, color="#333")
    for a, b in [("plan", "memory_read"), ("memory_read", "agent_answer"), ("agent_answer", "critic"),
                 ("critic", "memory_write"), ("memory_write", "stop")]:
        ax.annotate("", xy=(nodes[b][0] - 0.8, 2), xytext=(nodes[a][0] + 0.8, 2), arrowprops=arrow)
    ax.annotate("", xy=(5.6, 0.82), xytext=(6.2, 1.62), arrowprops=dict(arrow, color="#c92a2a"))
    ax.annotate("", xy=(4.5, 1.62), xytext=(5.0, 0.82), arrowprops=dict(arrow, color="#c92a2a"))
    ax.text(6.35, 1.25, "invalid", color="#c92a2a", fontsize=8); ax.text(7.05, 2.08, "valid", color="#2b8a3e", fontsize=8)
    ax.annotate("", xy=(10.0, 2.4), xytext=(6.3, 2.4), arrowprops=dict(arrowstyle="->", lw=1, ls="--", color="#868e96"),
                annotation_clip=False)
    ax.annotate("", xy=(9.6, 1.62), xytext=(6.6, 1.62), arrowprops=dict(arrowstyle="->", lw=1, ls="--", color="#868e96"))
    ax.text(8.0, 2.75, "invalid after retry  /  budget exhausted (any node) -> stop", ha="center", fontsize=8, color="#868e96")
    ax.set_xlim(-0.3, 10.9); ax.set_ylim(-0.1, 3.0)
    plt.tight_layout(); plt.savefig(path, dpi=130); plt.show()
draw_graph(RES / "01_graph.png")
app_graph = graph.build_graph(executor=agent.build_agent(LLM, verbose=False))
print(app_graph.get_graph().draw_mermaid())

## Error-recovery case (CLI, verbose)

In [ ]:
cli("reset-memory")                     # start from empty memory so this shows pure recovery
rec_txt = cli("ask", "Read policy remote_work_v9.9 and answer the limit.", "--max-steps", "8",
              title="Error recovery: bad doc_id -> soft error -> policy_search -> read_policy", png="05_error_recovery.png")

## 2.1 Memory: cold run vs warm run (`python app.py eval --cold`, then `python app.py eval`)

- **Writes:** only critic-approved answers whose `[doc_id]` exists and whose quotes are verbatim in the cited policy.
- **Reads:** the closest stored question with similarity ≥ 0.80. When several qualify, the cheapest one wins (fewest doc_ids to re-verify). Its answer and doc_id are passed into the question as a hint, so the agent can confirm with one `read_policy` call instead of searching.

In [ ]:
cold_txt = cli("eval", "--cold", "--quiet", title="Cold run: memory cleared, 8 canned questions", png="02_cold_run.png")

In [ ]:
warm_txt = cli("eval", "--quiet", title="Warm run: same questions, memory kept (memory hits)", png="03_warm_run.png")

In [ ]:
sfx = "_offline" if OFFLINE else ""
cold = pd.read_csv(agent.RUN_DIR / f"eval_cold{sfx}.csv"); warm = pd.read_csv(agent.RUN_DIR / f"eval_warm{sfx}.csv")
cmp = pd.DataFrame({"cold": [cold.latency_s.mean(), cold.agent_steps.mean(), cold.tokens.mean(), cold.PASS.mean(), int(cold.memory_hit.sum()), int(cold.memory_written.sum())],
                    "warm": [warm.latency_s.mean(), warm.agent_steps.mean(), warm.tokens.mean(), warm.PASS.mean(), int(warm.memory_hit.sum()), int(warm.memory_written.sum())]},
                   index=["avg latency (s)", "avg tool steps", "avg tokens", "pass rate", "memory hits", "memory writes"]).round(3)
cmp.to_csv(RES / "cold_vs_warm.csv")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, lab in zip(axes, ["agent_steps", "latency_s", "tokens"], ["tool steps", "latency (s)", "tokens"]):
    x = range(len(cold)); ax.bar([i - .2 for i in x], cold[col], .4, label="cold", color="#4e79a7")
    ax.bar([i + .2 for i in x], warm[col], .4, label="warm (memory)", color="#59a14f")
    ax.set_xticks(list(x)); ax.set_xticklabels(cold["id"]); ax.set_title(f"{lab} per question"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RES / "04_cold_vs_warm.png", dpi=130); plt.show()
cmp

## 2.2 (continued) Budget stop and the single retry

These two cases use controlled inputs so that both routing paths can be seen deterministically:
- **Budget stop:** the same question with `max_steps = 3`.
- **Critic failure + retry:** the agent is a scripted model that returns an uncited answer twice. This simulates a misbehaving model to prove the router retries exactly once and then stops. A real model rarely fails twice.

In [ ]:
mem_tmp = memory.FactMemory(agent.RUN_DIR / "memory_demo"); mem_tmp.clear()
b = graph.run_graph("Is MFA required for VPN? Cite and quote.",
                    app=graph.build_graph(executor=agent.build_agent(LLM, verbose=False), memory=mem_tmp), max_steps=3)
print("BUDGET STOP  status:", b["status"]); print("\n".join(b["trace"]))

from langchain_core.messages import AIMessage
bad = OfflinePolicyLLM(script=[AIMessage(content="Remote work is unlimited."), AIMessage(content="Still unlimited, trust me.")])
r = graph.run_graph("What is our remote work limit?", app=graph.build_graph(executor=agent.build_agent(bad, verbose=False), memory=mem_tmp))
print("\nCRITIC RETRY  status:", r["status"], "| retries:", r["retries"], "| memory size:", mem_tmp.size())
print("\n".join(r["trace"]))
term_png("BUDGET STOP (max_steps=3)\n" + "\n".join(b["trace"]) + f"\nstatus = {b['status']}\n\n"
         "CRITIC FAILURE -> ONE RETRY -> STOP (scripted bad model)\n" + "\n".join(r["trace"]) + f"\nstatus = {r['status']}, memory writes = 0",
         RES / "06_budget_and_retry.png", "Graph routing: budget stop and single retry")

## 2.3 Critic: minimal JSON verdicts

In [ ]:
for a in ["Remote work is allowed 3 days a week.",
          '"Employees may work remotely every day" [remote_work_v2]',
          'You get 30 days. "Full-time employees receive 24 days of annual leave per calendar year" [leave_v3]',
          '"Employees could work remotely for a maximum of 2 days per week" [remote_work_v1]',
          '"Employees may work remotely for a maximum of 3 days per week" [remote_work_v2]']:
    print(json.dumps(graph.critic_check("What is the policy?", a).model_dump()), "<-", a[:70])

## 2.4 Tests, SQLite checkpointing and the final evaluation table

In [ ]:
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "tests", "-p", "no:warnings"], capture_output=True, text=True)
print(t.stdout[-1500:])
term_png("$ pytest -q tests\n" + t.stdout[-2500:], RES / "07_pytest.png", "PyTest: tools, acceptance, graph budget")

In [ ]:
# Optional extension: SQLite checkpointing - every node's state is persisted and can be inspected later
ckpt = graph.sqlite_checkpointer(agent.RUN_DIR / "checkpoints.sqlite")
app_ck = graph.build_graph(executor=agent.build_agent(LLM, verbose=False), checkpointer=ckpt)
tid = f"demo-{int(time.time())}"
o = graph.run_graph("Is MFA required for VPN? Cite and quote.", app=app_ck, thread_id=tid)
hist = list(app_ck.get_state_history({"configurable": {"thread_id": tid}}))
print("status:", o["status"], "| checkpoints stored for this thread:", len(hist))

In [ ]:
final = warm.copy()
cols = ["id", "kind", "status", "latency_s", "agent_steps", "graph_steps", "has_docid", "has_quote", "cited", "critic_valid", "memory_hit", "PASS"]
fig, ax = plt.subplots(figsize=(14, 0.42 * len(final) + 1)); ax.axis("off")
tb = ax.table(cellText=final[cols].fillna("-").astype(str).values, colLabels=cols, loc="center", cellLoc="center")
tb.auto_set_font_size(False); tb.set_fontsize(8); tb.scale(1, 1.4)
for (r, c), cell in tb.get_celld().items():
    if r == 0: cell.set_facecolor("#1f3b57"); cell.set_text_props(color="white", weight="bold")
    elif cols[c] == "PASS": cell.set_facecolor("#d3f9d8" if final.iloc[r - 1]["PASS"] else "#ffe3e3")
ax.set_title(f"Evaluation table (warm run) - pass rate {final.PASS.mean():.0%}", weight="bold")
plt.tight_layout(); plt.savefig(RES / "08_eval_table.png", dpi=130); plt.show()
summary = {"v1_agent_only_pass": round(v1.PASS.mean(), 3), "cold_pass": round(cold.PASS.mean(), 3),
           "warm_pass": round(warm.PASS.mean(), 3), "tests": t.stdout.strip().splitlines()[-1],
           "model": "offline stand-in" if OFFLINE else agent.OPENAI_MODEL}
json.dump(summary, open(RES / "summary.json", "w"), indent=2); summary

## Summary

- **Grounding:** every factual answer cites `[doc_id]` tokens and quotes the rule verbatim. The critic verifies both against the corpus before anything is stored.
- **Agent behaviour:** the agent recovers from a bad doc_id by failing softly and then searching, combines the calculator with search, and refuses out-of-corpus questions. Iterations are capped at 6 per attempt.
- **Graph:** explicit nodes and routers enforce the step budget and a single retry after a critic failure. SQLite checkpointing makes every step inspectable.
- **Memory:** only verified facts are written. On the warm run, memory hits replace search with a single confirming read, which lowers steps, tokens and latency (see `results/04_cold_vs_warm.png`).